# Creating a Discussion

The simplest task you can accomplish with this library is to create a small discussion between LLMs. 

This guide will teach you the basic setup of the library. You will understand how to setup models, user-agents and how to coordinate them in a discussion. By the end of htis guide, you will be able to run a small discussion with a moderator and save it to the disk for persistence.

## Basics

### The Model

SynDisco can theoretically support any LLM, as long as it is wrapped in a `BaseModel` wrapper. The `BaseModel` class is a very simple interface with one method. This method gives the underlying LLM input, and returns its output to the library.

There already exists a `TransformersModel` class which handles models from the `transformers` python library. In 90% of your applications, this will be enough. We can load a TransformersModel using the following code:

In [2]:
import syndisco

llm = syndisco.TransformersModel(
    model_path="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    name="test_model",
    max_out_tokens=100,
)

This will download a small LLM from huggingface. You can substitute the model_path for any similar model in [HuggingFace](https://huggingface.co/) supporting the Transformers library.

### Creating personas

All `actors` can be defined by a `persona`, aka a set of attributes that define them. These attributes can be age, ethnicity, and even include special instructions on how they should behave.

Syndisco uses any python dictionary for defining personas. These dictionaries are given as-is to the underlying prompt, and saved as JSON in the exported discussion logs.

In [3]:
personas = [
    {
        "username": "Gwx31",
        "age": 38,
        "sex": "female",
        "education_level": "Bachelor's",
        "sexual_orientation": "Heterosexual",
        "demographic_group": "Latino",
        "current_employment": "Registered Nurse",
        "personality_characteristics": [
            "compassionate",
            "patient",
            "diligent",
            "overwhelmed",
        ],
    },
    {
        "username": "Giannis",
        "age": 21,
        "sex": "male",
        "education_level": "College",
        "sexual_orientation": "Pansexual",
        "demographic_group": "White",
        "current_employment": "Game Developer",
        "personality_characteristics": [
            "strategic",
            "meticulous",
            "nerdy",
            "hyper-focused",
        ],
    },
    {
        "username": "Kimya",
        "age": 67,
        "sex": "female",
        "education_level": "PhD",
        "sexual_orientation": "Heterosexual",
        "demographic_group": "White",
        "personality_characteristics": [
            "strict",
            "grumpy"
        ],
    }
]

Since creating a lot of distinct users is essential in running large-scale experiments, users are usually defined in JSON format. That way, you can change anything without touching your code!

[Here](https://github.com/dimits-ts/synthetic_moderation_experiments/blob/master/data/discussions_input/personas/personas.json) is an applied example of how to mass-define user personas through JSON files.

### Creating the user-agents

Having a `persona` and a `model` we can finally create an `actor`. The actor will personify the selected persona using the model to talk.

Besides a persona and a model, the actors will also need instructions and a context. By convention, all actors share the same context, and all user-agents share the same instructions. Personalized instructions are defined in the actor's persona.

In [4]:
CONTEXT = "You are taking part in an online conversation"
INSTRUCTIONS = "Act like a human would"

actors = [
    syndisco.Actor(
        model=llm,
        persona=p,
        context=CONTEXT,
        instructions=INSTRUCTIONS,
        name=p["username"]
    )
    for p in personas
]

### Managing turn-taking

In real-life discussions, who gets to speak at each point in time is determined by complex social dynamics, which are difficult to realistically model.
However, there are ways with which we can simulate these dynamics. 

SynDisco uses the `TurnManager` class to model turn taking. Three implementations are available by default:

- `QueueTurnManager` is the simplest, most intuitive way to model turn-taking; everyone gets to talk once per round. Once everyone talks once, they get to talk again in the same sequence

In [5]:
rrobin_turn_manager = syndisco.QueueTurnManager(actors)

for i in range(10):
    speaker = next(rrobin_turn_manager)
    print(speaker.get_actor_name())

Gwx31
Giannis
Kimya
Gwx31
Giannis
Kimya
Gwx31
Giannis
Kimya
Gwx31


- `RandomTurnManager` selects a random speaker each time (excluding the last speaker):

In [6]:
rrobin_turn_manager = syndisco.RandomTurnManager(actors)

for i in range(10):
    speaker = next(rrobin_turn_manager)
    print(speaker.get_actor_name())

Gwx31
Giannis
Gwx31
Kimya
Gwx31
Kimya
Gwx31
Giannis
Kimya
Giannis


/media/SSD_4TB_2/dtsirmpas/projects/syndisco/src/syndisco/turn_manager.py:142: UserWarning: p_respond has been set to 0, which disables responding altogether. In that case, it may be better to use the RandomTurnManager class instead.
  warnings.warn(


- `RespondTurnManager` on the other hand throws a weighted coin on each round. If the coin flip succedes, the previous user gets to respond. If not, another user is selected at random

In [7]:
res_turn_manager = syndisco.RespondTurnManager(actors, p_respond=0.5)

for i in range(10):
    speaker = next(res_turn_manager)
    print(speaker.get_actor_name())

Gwx31
Kimya
Gwx31
Kimya
Gwx31
Giannis
Gwx31
Giannis
Kimya
Giannis


## Generating a discussion

Let's start with the most basic task; a single discussion between two users. Let's also add a few starting comments, supposedly made by the first user, to get the ball rolling.

In [8]:
turn_manager = syndisco.QueueTurnManager(actors)
conv = syndisco.Discussion(
    next_turn_manager=turn_manager,
    users=actors,
    seed_opinions=["I hate the concept of chairs."],
    seed_opinion_usernames=[personas[0]["username"]]
)
conv.begin()

  0%|          | 0/5 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Comment by user Gwx31: "
Oh, wow, that's quite a statement! I can understand where you're coming from. Sometimes sitting for long periods can be really uncomfortable, especially if you have a job that requires a lot of sitting. As a nurse, I can relate to the physical strain it can put on your body. Maybe finding ways to move around more or using ergonomic chairs could help make things a bit better? What inspired your dislike of chairs specifically?" 

Comment by user Giannis: "
As a game developer who spends most of my day in front of a screen, I can definitely relate to the discomfort that comes with prolonged sitting. However, my issue with chairs isn't necessarily about the physical discomfort—it's more about the idea of them as a societal norm and the expectations they impose. Chairs often symbolize a certain level of formality and permanence in our workspaces, which can feel restrictive. Plus, there's something about the fluidity of movement that just feels more natural to" 

Com

## Driving a discussion step by step

`Discussion` implements Python's iterator protocol, so you are not limited
to running the whole conversation at once with `begin()`. You can drive it
one turn at a time, which is useful when you want to inspect or react to
each entry before the next speaker is prompted — for example to apply
early stopping, live logging, or conditional branching.

Because `Discussion` is its own iterator (`__iter__` returns `self`), it
is **single-pass**: once `StopIteration` is raised the instance is
exhausted and should not be reused.

### Calling `next()` manually

Calling `iter()` on a `Discussion` performs the one-time setup (registering
participant names and inserting any seed opinions). Each subsequent `next()`
prompts one speaker and returns the new log entry as a plain dict.

In [9]:
turn_manager = syndisco.QueueTurnManager(actors)
conv = syndisco.Discussion(
    next_turn_manager=turn_manager,
    users=actors,
    conv_len=4,
    seed_opinions=["I hate the concept of chairs."],
    seed_opinion_usernames=[personas[0]["username"]],
)

it = iter(conv)  # one-time setup happens here

first_entry = next(it)
print("First speaker: ", first_entry["name"])
print("Their message: ", first_entry["text"], "...")

second_entry = next(it)
print("\nSecond speaker:", second_entry["name"])
print("Their message: ", second_entry["text"], "...")

First speaker:  Gwx31
Their message:  Oh, wow, that's quite a statement! I can understand where you're coming from. Sometimes sitting for long periods can be really uncomfortable, especially if you have a job that requires a lot of sitting. As a nurse, I can relate to the physical strain it can put on your body. Maybe finding ways to move around more or using ergonomic chairs could help make things a bit better? What inspired your dislike of chairs specifically? ...

Second speaker: Giannis
Their message:  As a game developer who spends most of my day in front of a screen, I can definitely relate to the discomfort that comes with prolonged sitting. However, my issue with chairs isn't necessarily about the physical discomfort—it's more about the idea of them as a societal norm and the expectations they impose. Chairs often symbolize a certain level of formality and permanence in our workspaces, which can feel restrictive. Plus, there's something about the fluidity of movement that just 

### Using a `for` loop

The same iterator works directly in a `for` loop. Here we stop early if
a particular word appears in an entry - something that is impossible with
`begin()`.

In [10]:
turn_manager = syndisco.QueueTurnManager(actors)
conv = syndisco.Discussion(
    next_turn_manager=turn_manager,
    users=actors,
    conv_len=10,
    seed_opinions=["I hate the concept of chairs."],
    seed_opinion_usernames=[personas[0]["username"]],
)

for entry in conv:
    print(entry["text"])

print(f"\nTurns completed so far: {conv._steps_taken}")

Oh, wow, that's quite a statement! I can understand where you're coming from. Sometimes sitting for long periods can be really uncomfortable, especially if you have a job that requires a lot of sitting. As a nurse, I can relate to the physical strain it can put on your body. Maybe finding ways to move around more or using ergonomic chairs could help make things a bit better? What inspired your dislike of chairs specifically?
As a game developer who spends most of my day in front of a screen, I can definitely relate to the discomfort that comes with prolonged sitting. However, my issue with chairs isn't necessarily about the physical discomfort—it's more about the idea of them as a societal norm and the expectations they impose. Chairs often symbolize a certain level of formality and permanence in our workspaces, which can feel restrictive. Plus, there's something about the fluidity of movement that just feels more natural to
As Kimya, I might respond with something like this:

"Interes

Congratulations, you can now run fully synthetic discussions! You may want to experiment with adding more than 2 users or testing more realistic turn taking procedures (for example, check out the `RandomWeighted` turn manager).

## Working with `Logs`

All log state is held in `Discussion.logs`, an instance of `Logs`.
This object supports the standard sequence interface, so you can index,
slice, and iterate over it directly - no need to call `to_dict()` just to
inspect entries.

In [11]:
logs = conv.get_logs()

print(f"Total entries : {len(logs)}")
print(f"First entry   : {logs[0]}")
print(f"Last entry    : {logs[-1]}")
print()

# Iterate directly — each item is a dict with 'name', 'text', 'model'
for entry in logs:
    print(f"[{entry['name']}] {entry['text'][:60]}...")

Total entries : 11
First entry   : {'name': 'Gwx31', 'text': 'I hate the concept of chairs.', 'model': 'hardcoded', 'prompt': '{"context": "", "instructions": "", "type": "user", "persona": {}}'}
Last entry    : {'name': 'Gwx31', 'text': 'Thank you all for the kind words and for sharing such insightful perspectives! It’s amazing how different professions can offer such varied insights into the role of chairs in our daily lives. As a registered nurse, I can definitely relate to both the physical and mental aspects of working in a chair for long periods. It’s great to see that others are also thinking about how to balance comfort and productivity.\n\nFor me, as a nurse, the physical demands of the job mean that finding ways to move around and stay', 'model': 'test_model', 'prompt': '{"context": "You are taking part in an online conversation", "instructions": "Act like a human would", "type": "user", "persona": {"username": "Gwx31", "age": 38, "sex": "female", "education_level": "Bachelor

### Exporting and importing discussions with `Logs`

In [12]:
import json
import tempfile


tp = tempfile.NamedTemporaryFile(suffix=".json", delete=False)
logs = conv.get_logs()
logs.export(tp.name)

with open(tp.name) as f:
    data = json.load(f)

print(json.dumps(data, indent=4))

{
    "timestamp": "26-06-11-13-47",
    "logs": [
        {
            "name": "Gwx31",
            "text": "I hate the concept of chairs.",
            "model": "hardcoded",
            "prompt": "{\"context\": \"\", \"instructions\": \"\", \"type\": \"user\", \"persona\": {}}"
        },
        {
            "name": "Gwx31",
            "text": "Oh, wow, that's quite a statement! I can understand where you're coming from. Sometimes sitting for long periods can be really uncomfortable, especially if you have a job that requires a lot of sitting. As a nurse, I can relate to the physical strain it can put on your body. Maybe finding ways to move around more or using ergonomic chairs could help make things a bit better? What inspired your dislike of chairs specifically?",
            "model": "test_model",
            "prompt": "{\"context\": \"You are taking part in an online conversation\", \"instructions\": \"Act like a human would\", \"type\": \"user\", \"persona\": {\"username\":

In [13]:
logs2 = syndisco.Logs.from_file(tp.name)
print(logs2)

{
    "timestamp": "26-06-11-13-47",
    "logs": [
        {
            "name": "Gwx31",
            "text": "I hate the concept of chairs.",
            "model": "hardcoded",
            "prompt": ""
        },
        {
            "name": "Gwx31",
            "text": "Oh, wow, that's quite a statement! I can understand where you're coming from. Sometimes sitting for long periods can be really uncomfortable, especially if you have a job that requires a lot of sitting. As a nurse, I can relate to the physical strain it can put on your body. Maybe finding ways to move around more or using ergonomic chairs could help make things a bit better? What inspired your dislike of chairs specifically?",
            "model": "test_model",
            "prompt": ""
        },
        {
            "name": "Giannis",
            "text": "As a game developer who spends most of my day in front of a screen, I can definitely relate to the discomfort that comes with prolonged sitting. However, my issue 

### Creating hard-coded logs

The `Logs` class also enables you to hardcode the data inside. This would be useful for example if you want to use `syndisco`'s annotation on data that were not generated by the `Discussion` API (see next guide).

In [14]:
names=["Alice", "Bob", "Alice"]
texts=[
    "What do you think about the new policy?",
    "Honestly, I think it goes too far.",
    "I can see that perspective, though I'm not sure I agree.",
]
handcrafted = syndisco.Logs()

for name, text in zip(names, texts):
    handcrafted.append(name, text)

print(handcrafted)

{
    "timestamp": "26-06-11-13-47",
    "logs": [
        {
            "name": "Alice",
            "text": "What do you think about the new policy?",
            "model": "hardcoded",
            "prompt": ""
        },
        {
            "name": "Bob",
            "text": "Honestly, I think it goes too far.",
            "model": "hardcoded",
            "prompt": ""
        },
        {
            "name": "Alice",
            "text": "I can see that perspective, though I'm not sure I agree.",
            "model": "hardcoded",
            "prompt": ""
        }
    ]
}
